<!-- # core

> Fill in a module description here -->

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| exports

from fastcore.basics import *
from fastcore.meta import *
from fastcore.test import *
import json
from rdflib import Graph
from pyld import jsonld
from typing import List, Dict, Any, Optional, Union

In [ ]:
#| hide
from IPython.display import display, Markdown, HTML

In [ ]:
#| export
class LinkedDataKnowledge:
    "Represents a knowledge base of linked data in JSON-LD format"
    def __init__(self, 
                 data:Dict=None, # Initial knowledge data
                ):
        self.data = data or {"@context": {}, "@graph": []}
        
    def __repr__(self): return f"LinkedDataKnowledge with {len(self.data.get('@graph', []))} entities"

In [ ]:
#| export
@patch
def _repr_markdown_(self:LinkedDataKnowledge) -> str:
    "Rich markdown representation for notebook display"
    md = [f"## LinkedDataKnowledge"]
    
    # Context summary
    context = self.data.get('@context', {})
    md.append(f"### Context ({len(context)} prefixes)")
    
    # Show the first few context entries
    if context:
        md.append("```json")
        context_preview = dict(list(context.items())[:5])
        if len(context) > 5:
            md.append(json.dumps(context_preview, indent=2) + "\n... and more")
        else:
            md.append(json.dumps(context, indent=2))
        md.append("```")
    
    # Graph summary
    graph = self.data.get('@graph', [])
    md.append(f"### Graph ({len(graph)} entities)")
    
    # Show types of entities in the graph
    if graph:
        types = {}
        for entity in graph:
            entity_type = entity.get('@type')
            if isinstance(entity_type, list):
                for t in entity_type:
                    types[t] = types.get(t, 0) + 1
            elif entity_type:
                types[entity_type] = types.get(entity_type, 0) + 1
        
        if types:
            md.append("**Entity types:**")
            for t, count in sorted(types.items(), key=lambda x: x[1], reverse=True):
                md.append(f"- {t}: {count}")
    
    # Show a sample entity if available
    if graph:
        md.append("\n**Sample entity:**")
        md.append("```json")
        md.append(json.dumps(graph[0], indent=2))
        md.append("```")
    
    # If using @included, show that too
    if '@included' in self.data:
        included = self.data['@included']
        md.append(f"\n### Included ({len(included)} entities)")
        md.append("**Types of included entities:**")
        
        # Count types of included entities
        included_types = {}
        for entity in included:
            entity_type = entity.get('@type')
            if isinstance(entity_type, list):
                for t in entity_type:
                    included_types[t] = included_types.get(t, 0) + 1
            elif entity_type:
                included_types[entity_type] = included_types.get(entity_type, 0) + 1
        
        for t, count in sorted(included_types.items(), key=lambda x: x[1], reverse=True):
            md.append(f"- {t}: {count}")
    
    return "\n".join(md)


In [ ]:
#| export
def _format_entity_markdown(entity:Dict) -> str:
    "Format a single entity as markdown"
    md = []
    
    # Entity ID and type
    entity_id = entity.get('@id', 'No ID')
    entity_type = entity.get('@type', ['Unknown'])
    if isinstance(entity_type, list):
        entity_type = ', '.join(entity_type)
    
    md.append(f"### {entity_type}: {entity_id}")
    
    # Properties
    for prop, values in entity.items():
        if prop in ['@id', '@type']:
            continue
            
        # Format the property name
        prop_name = prop.split('/')[-1] if '/' in prop else prop
        md.append(f"**{prop_name}**:")
        
        # Format values
        for val in values:
            if isinstance(val, dict):
                if '@value' in val:
                    value_text = val['@value']
                    if '@language' in val:
                        value_text += f" @{val['@language']}"
                    md.append(f"- {value_text}")
                elif '@id' in val:
                    md.append(f"- [{val['@id']}]({val['@id']})")
                else:
                    md.append(f"- {val}")
            else:
                md.append(f"- {val}")
    
    return "\n".join(md)

@patch
def query_markdown(self:LinkedDataKnowledge,
                  query_type:str, # Type of query: "property", "type", "value" 
                  query_value:str, # Value to search for
                  ) -> str:
    "Query the knowledge base and return results as markdown"
    results = self.query(query_type, query_value)
    
    if not results:
        return f"*No results found for {query_type}='{query_value}'*"
    
    md = [f"# Query Results: {query_type}='{query_value}'", 
          f"Found {len(results)} matching entities"]
    
    # Format each result entity
    for i, entity in enumerate(results[:5]):  # Limit to first 5 for readability
        md.append(f"\n## Result {i+1}")
        md.append(_format_entity_markdown(entity))
    
    if len(results) > 5:
        md.append(f"\n*...and {len(results)-5} more results*")
    
    return "\n".join(md)


In [ ]:
#| export
@patch
def display_entity(self:LinkedDataKnowledge, entity_id:str) -> str:
    "Display a specific entity by ID as markdown"
    
    # First try in @graph
    for entity in self.data.get('@graph', []):
        if entity.get('@id') == entity_id:
            return _format_entity_markdown(entity)
    
    # Then try in @included if present
    for entity in self.data.get('@included', []):
        if entity.get('@id') == entity_id:
            return _format_entity_markdown(entity)
    
    # If it's the main entity in a resource-centric view
    if self.data.get('@id') == entity_id:
        return _format_entity_markdown(self.data)
    
    return f"*Entity with ID '{entity_id}' not found*"

@patch
def summarize_markdown(self:LinkedDataKnowledge) -> str:
    "Provide a concise markdown summary of the knowledge base"
    
    md = ["# Knowledge Base Summary"]
    
    # Count contexts, entities, and included entities
    context_count = len(self.data.get('@context', {}))
    graph_count = len(self.data.get('@graph', []))
    included_count = len(self.data.get('@included', [])) if '@included' in self.data else 0
    
    md.append(f"- **Contexts:** {context_count}")
    md.append(f"- **Graph Entities:** {graph_count}")
    if included_count:
        md.append(f"- **Included Entities:** {included_count}")
    
    # Summarize entity types
    all_types = {}
    
    # Check @graph
    for entity in self.data.get('@graph', []):
        entity_type = entity.get('@type')
        if isinstance(entity_type, list):
            for t in entity_type:
                all_types[t] = all_types.get(t, 0) + 1
        elif entity_type:
            all_types[entity_type] = all_types.get(entity_type, 0) + 1
    
    # Check @included
    for entity in self.data.get('@included', []):
        entity_type = entity.get('@type')
        if isinstance(entity_type, list):
            for t in entity_type:
                all_types[t] = all_types.get(t, 0) + 1
        elif entity_type:
            all_types[entity_type] = all_types.get(entity_type, 0) + 1
    
    # Check main entity if resource-centric
    if '@id' in self.data and '@type' in self.data:
        entity_type = self.data.get('@type')
        if isinstance(entity_type, list):
            for t in entity_type:
                all_types[t] = all_types.get(t, 0) + 1
        elif entity_type:
            all_types[entity_type] = all_types.get(entity_type, 0) + 1
    
    if all_types:
        md.append("\n## Entity Types")
        for t, count in sorted(all_types.items(), key=lambda x: x[1], reverse=True)[:10]:
            md.append(f"- **{t}**: {count}")
        if len(all_types) > 10:
            md.append(f"- *...and {len(all_types)-10} more types*")
    
    return "\n".join(md)


In [ ]:
@patch
def find_entity(self:LinkedDataKnowledge, 
               entity_id:str=None, # Full or partial entity ID to find
               term_type:str=None, # Filter by type (e.g., "Class", "Property")
               label:str=None # Find by label text
              ) -> list: # Matching entities
    "Find entities in the graph by ID, type, or label"
    results = []
    graph = self.data.get('@graph', [])
    
    for entity in graph:
        # Match by ID if specified
        if entity_id and isinstance(entity.get('@id'), str):
            if entity_id in entity['@id']:
                if term_type is None or self._has_type(entity, term_type):
                    results.append(entity)
                    continue
        
        # Match by label if specified
        if label and not entity_id:
            for key, value in entity.items():
                if 'label' in key.lower():
                    if isinstance(value, list):
                        for item in value:
                            if isinstance(item, dict) and item.get('@value') == label:
                                if term_type is None or self._has_type(entity, term_type):
                                    results.append(entity)
                                    break
                    elif isinstance(value, str) and label in value:
                        if term_type is None or self._has_type(entity, term_type):
                            results.append(entity)
                            break
    
    return results

In [ ]:
# Then create a test function that doubles as documentation
def test_find_entity():
    "Test finding entities by ID, type, and label"
    # Create a test LinkedDataKnowledge object
    data = {
        '@graph': [
            {'@id': 'http://example.org/entity1', 'rdfs:label': 'Entity One'},
            {'@id': 'http://example.org/entity2', 'rdfs:label': 'Entity Two'},
            {'@id': 'http://example.org/entity3', 'rdfs:label': 'Entity Three', '@type': 'Class'},
        ]
    }
    
    kb = LinkedDataKnowledge(data)
    
    # Test finding by ID - this serves as documentation example too
    result1 = kb.find_entity(entity_id='entity1')
    test_eq(len(result1), 1)
    test_eq(result1[0]['@id'], 'http://example.org/entity1')
    
    # Test finding by partial ID
    result2 = kb.find_entity(entity_id='entity')
    test_eq(len(result2), 3)
    
    # Test finding by label
    result3 = kb.find_entity(label='Entity Two')
    test_eq(len(result3), 1)
    test_eq(result3[0]['@id'], 'http://example.org/entity2')
    
    # Test filtering by type
    result4 = kb.find_entity(entity_id='entity', term_type='Class')
    test_eq(len(result4), 1)
    test_eq(result4[0]['@id'], 'http://example.org/entity3')
    
    # Test with non-existent entity
    result5 = kb.find_entity(entity_id='nonexistent')
    test_eq(len(result5), 0)
    
    # You can also display a rich representation for documentation
    from IPython.display import display, Markdown
    display(Markdown("### Sample entity found:"))
    if result1:
        display(Markdown(f"**ID**: {result1[0]['@id']}  \n**Label**: {result1[0]['rdfs:label']}"))


In [ ]:
test_find_entity()

### Sample entity found:

**ID**: http://example.org/entity1  
**Label**: Entity One

In [ ]:
@patch
def _has_type(self:LinkedDataKnowledge, entity:dict, type_str:str) -> bool:
    "Check if entity has the specified type"
    entity_type = entity.get('@type', [])
    if not isinstance(entity_type, list):
        entity_type = [entity_type]
    
    return any(type_str in t for t in entity_type)

In [ ]:
@patch
def get_entity_description(self:LinkedDataKnowledge, entity:dict) -> str:
    "Get a formatted description of an entity"
    if not entity:
        return "No entity provided"
    
    lines = []
    
    # Entity ID
    entity_id = entity.get('@id', 'Unknown ID')
    lines.append(f"## Entity: {entity_id}")
    
    # Entity type
    entity_type = entity.get('@type', [])
    if not isinstance(entity_type, list):
        entity_type = [entity_type]
    lines.append(f"**Type**: {', '.join(entity_type)}")
    
    # Labels
    labels = []
    for key, value in entity.items():
        if 'label' in key.lower():
            if isinstance(value, list):
                for item in value:
                    if isinstance(item, dict) and '@value' in item:
                        labels.append(f"{item.get('@value')} ({item.get('@language', 'no language')})")
                    else:
                        labels.append(str(item))
            else:
                labels.append(str(value))
    
    if labels:
        lines.append(f"**Labels**: {', '.join(labels)}")
    
    # Comments/Definitions
    comments = []
    for key, value in entity.items():
        if 'comment' in key.lower() or 'definition' in key.lower():
            if isinstance(value, list):
                for item in value:
                    if isinstance(item, dict) and '@value' in item:
                        comments.append(item.get('@value'))
                    else:
                        comments.append(str(item))
            else:
                comments.append(str(value))
    
    if comments:
        lines.append("\n**Definition**:")
        for comment in comments:
            lines.append(f"- {comment}")
    
    # Relationships
    relationships = []
    for key, value in entity.items():
        if key not in ['@id', '@type'] and not any(x in key.lower() for x in ['label', 'comment', 'definition']):
            relationships.append((key, value))
    
    if relationships:
        lines.append("\n**Relationships**:")
        for rel_name, rel_value in relationships:
            if isinstance(rel_value, list):
                lines.append(f"- {rel_name}:")
                for item in rel_value:
                    if isinstance(item, dict) and '@id' in item:
                        lines.append(f"  - {item['@id']}")
                    elif isinstance(item, dict) and '@value' in item:
                        lines.append(f"  - {item['@value']}")
                    else:
                        lines.append(f"  - {item}")
            else:
                if isinstance(rel_value, dict) and '@id' in rel_value:
                    lines.append(f"- {rel_name}: {rel_value['@id']}")
                elif isinstance(rel_value, dict) and '@value' in rel_value:
                    lines.append(f"- {rel_name}: {rel_value['@value']}")
                else:
                    lines.append(f"- {rel_name}: {rel_value}")
    
    return "\n".join(lines)


In [ ]:
@patch
def query(self:LinkedDataKnowledge,
         query_type:str, # Type of query: "property", "type", "value"
         query_value:str # Value to search for
         ) -> list:
    """Queries JSON-LD data for specific properties, types, or values."""
    # First, try to expand the query value if it's a prefixed term
    expanded_value = query_value
    
    # Handle prefixed terms (like schema:Person)
    if ':' in query_value and not query_value.startswith(('http://', 'https://')):
        prefix, local = query_value.split(':', 1)
        if prefix in self.data.get('@context', {}):
            prefix_uri = self.data['@context'][prefix]
            if isinstance(prefix_uri, str):
                if prefix_uri.endswith('/') or prefix_uri.endswith('#'):
                    expanded_value = f"{prefix_uri}{local}"
                else:
                    expanded_value = f"{prefix_uri}/{local}"
    
    # Handle non-prefixed terms that might be defined in context
    elif not query_value.startswith(('http://', 'https://')):
        # Check if term is defined in @context
        if query_value in self.data.get('@context', {}):
            term_def = self.data['@context'][query_value]
            if isinstance(term_def, dict) and '@id' in term_def:
                expanded_value = term_def['@id']
            elif isinstance(term_def, str):
                expanded_value = term_def
        
        # Check if there's a @vocab that would apply
        elif '@vocab' in self.data.get('@context', {}):
            vocab = self.data['@context']['@vocab']
            if vocab.endswith('/') or vocab.endswith('#'):
                expanded_value = f"{vocab}{query_value}"
            else:
                expanded_value = f"{vocab}/{query_value}"
    
    # Convert JSON-LD to expanded form for consistent access
    expanded = jsonld.expand(self.data)
    results = []
    
    if query_type == "property":
        # Find entities with a specific property (using both original and expanded)
        for entity in expanded:
            if query_value in entity or expanded_value in entity:
                results.append(entity)
    elif query_type == "type":
        # Find entities of a specific type (using both original and expanded)
        for entity in expanded:
            if '@type' not in entity:
                continue
                
            types = entity['@type']
            if not isinstance(types, list):
                types = [types]
                
            if query_value in types or expanded_value in types:
                results.append(entity)
    elif query_type == "value":
        # Find entities with properties containing a specific value
        for entity in expanded:
            for prop, values in entity.items():
                if prop == "@id" or prop == "@type":
                    continue
                for val in values:
                    if "@value" in val and val["@value"] == query_value:
                        results.append(entity)
                        break
    
    return results


In [ ]:
 #| export
def describe(self, path:str="") -> str:
    "Describe the structure at the given path in a human-readable format"
    # Implementation that combines exploration and visualization
    pass

In [ ]:
#| export
def view(self, entity_id:str=None, term_type:str=None, label:str=None) -> None:
    "Find and display entities in a rich format"
    entities = self.find_entity(entity_id, term_type, label)
    if not entities:
        print(f"No entities found matching the criteria")
        return
    
    for entity in entities:
        display(Markdown(self.get_entity_description(entity)))

In [ ]:
#| test
def test_initialization():
    kb = LinkedDataKnowledge()
    test_eq(kb.data, {"@context": {}, "@graph": []})
    
    test_data = {"@context": {"schema": "https://schema.org/"}, "@graph": [{"@id": "test"}]}
    kb2 = LinkedDataKnowledge(test_data)
    test_eq(kb2.data, test_data)

In [ ]:
test_initialization()

In [ ]:
#| test
def test_find_entity():
    test_kb = LinkedDataKnowledge({
        "@context": {},
        "@graph": [
            {
                "@id": "http://example.org/Person",
                "@type": "rdfs:Class",
                "rdfs:label": "Person"
            },
            {
                "@id": "http://example.org/name",
                "@type": "rdf:Property",
                "rdfs:label": "name"
            }
        ]
    })
    # Test find by ID
    person_results = test_kb.find_entity(entity_id="Person")
    test_eq(len(person_results), 1)
    test_eq(person_results[0]['@id'], "http://example.org/Person")

    # Test find by type
    class_results = test_kb.find_entity(term_type="rdfs:Class")
    test_eq(len(class_results), 1)
    test_eq(class_results[0]['@id'], "http://example.org/Person")

    # Test find by label
    label_results = test_kb.find_entity(label="Person")
    test_eq(len(label_results), 1)
    test_eq(label_results[0]['@id'], "http://example.org/Person")

In [ ]:
def test_find_entity():
    # Create a test LinkedDataKnowledge object
    data = {
        '@graph': [
            {'@id': 'http://example.org/entity1', 'rdfs:label': 'Entity One'},
            {'@id': 'http://example.org/entity2', 'rdfs:label': 'Entity Two'},
            {'@id': 'http://example.org/entity3', 'rdfs:label': 'Entity Three', '@type': 'Class'},
        ]
    }
    
    knowledge = LinkedDataKnowledge(data)
    
    # Test finding by ID
    result1 = knowledge.find_entity(entity_id='entity1')
    assert len(result1) == 1
    assert result1[0]['@id'] == 'http://example.org/entity1'
    
    # Test finding by partial ID
    result2 = knowledge.find_entity(entity_id='entity')
    assert len(result2) == 3
    
    # Test finding by label
    result3 = knowledge.find_entity(label='Entity Two')
    assert len(result3) == 1
    assert result3[0]['@id'] == 'http://example.org/entity2'
    
    # Test filtering by type
    result4 = knowledge.find_entity(entity_id='entity', term_type='Class')
    assert len(result4) == 1
    assert result4[0]['@id'] == 'http://example.org/entity3'
    
    # Test with non-existent entity
    result5 = knowledge.find_entity(entity_id='nonexistent')
    assert len(result5) == 0

In [ ]:
#| test
def test_get_entity_description():
    data = {
        '@graph': [
            {
                '@id': 'http://example.org/entity1', 
                'rdfs:label': 'Test Entity',
                'rdfs:comment': 'This is a test entity',
                '@type': 'Class'
            },
            {
                '@id': 'http://example.org/entity2', 
                'rdfs:label': [{'@value': 'Multi-language Entity', '@language': 'en'}],
                'skos:definition': 'Entity with complex properties'
            }
        ]
    }
    
    kb = LinkedDataKnowledge(data)
    
    # Test basic description
    entity1 = kb.find_entity(entity_id='entity1')[0]
    desc1 = kb.get_entity_description(entity1)
    assert 'Test Entity' in desc1
    assert 'This is a test entity' in desc1
    assert 'Class' in desc1
    
    # Test with complex properties
    entity2 = kb.find_entity(entity_id='entity2')[0]
    desc2 = kb.get_entity_description(entity2)
    assert 'Multi-language Entity' in desc2
    assert 'Entity with complex properties' in desc2

In [ ]:
test_get_entity_description()

In [ ]:
test_find_entity()

In [ ]:
#| test
def test_markdown_representation():
    kb = LinkedDataKnowledge({
        "@context": {"schema": "https://schema.org/"},
        "@graph": [
            {
                "@id": "http://example.org/person1",
                "@type": "schema:Person",
                "schema:name": "Alice Smith"
            }
        ]
    })
    
    md = kb._repr_markdown_()
    assert('LinkedDataKnowledge' in md)
    assert('Context' in md)
    assert('Graph' in md)
    assert('schema:Person' in md)
    

In [ ]:
test_markdown_representation()

In [ ]:
from IPython.display import Markdown, display

def test_markdown_display():
    """Test the markdown display of entity descriptions"""
    data = {
        '@graph': [
            {
                '@id': 'http://example.org/entity1', 
                'rdfs:label': 'Test Entity',
                'rdfs:comment': 'This is a test entity',
                '@type': 'Class'
            }
        ]
    }
    
    kb = LinkedDataKnowledge(data)
    entity = kb.find_entity(entity_id='entity1')[0]
    desc = kb.get_entity_description(entity)
    
    # Display the markdown representation
    display(Markdown(desc))
    
    return "Markdown display test passed"

# Run the test
test_markdown_display()

## Entity: http://example.org/entity1
**Type**: Class
**Labels**: Test Entity

**Definition**:
- This is a test entity

'Markdown display test passed'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()